In [65]:
import torch
import tiktoken
from torch import nn

In [137]:
torch.manual_seed(123)

In [138]:
# Hyperparameters
config = {
    "context_length": 6,
    "dim_in": 4,
    "dim_out": 3
}

dim_in = 4
dim_out = 3
context_length = 6

In [139]:
class TokenizerV1:
    def __init__(self):
        self.tokenizer = tiktoken.get_encoding("gpt2")
        self.n_vocab = self.tokenizer.n_vocab

    def encode(self, text):
        return self.tokenizer.encode(text)

    def decode(self, encodings):
        return self.tokenizer.decode(encodings)

In [140]:
class InputPreprocessor(nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.token_embeddings = nn.Embedding(tokenizer.n_vocab, config["dim_in"])
        self.pos_embeddings = nn.Embedding(config["context_length"], config["dim_in"])

    def forward(self, text):
        encodings = self.tokenizer.encode(text)
        encodings = torch.tensor(encodings[:config["context_length"]])
        return self.token_embeddings(encodings) + self.pos_embeddings(torch.arange(0, config["context_length"]))
        

In [141]:
tokenizer = TokenizerV1()
preprocessor = InputPreprocessor(tokenizer)

In [142]:
input_embeddings = preprocessor("I want to play football.")

In [143]:
input_embeddings

tensor([[-0.6548,  2.1284, -0.2805,  1.3280],
        [ 0.9075, -0.3510, -0.6833, -2.9320],
        [ 0.3414,  0.5612, -0.7663,  1.4225],
        [ 2.4883, -0.4090,  2.2379,  0.2385],
        [-0.9753, -0.0234,  0.6451, -1.2915],
        [-0.9374,  0.3967,  0.8233, -0.6069]], grad_fn=<AddBackward0>)

In [144]:
W_keys = nn.Linear(dim_in, dim_out)
W_queries = nn.Linear(dim_in, dim_out)
W_values = nn.Linear(dim_in, dim_out)

In [145]:
keys = W_keys(input_embeddings)
queries = W_queries(input_embeddings)
values = W_values(input_embeddings)

In [146]:
queries @ keys.T # attention_scores

tensor([[-0.4681,  1.9656, -0.4303, -1.3786,  1.4623,  0.8969],
        [-0.1567, -3.1167,  0.4975,  1.3980, -1.8190, -1.2151],
        [-0.5642,  0.8800, -0.3636, -0.5320,  0.6212,  0.3152],
        [-0.4883,  2.0215, -0.2630, -2.1553,  1.9336,  1.2343],
        [-0.2234, -0.7167,  0.0564,  0.2360, -0.3600, -0.2697],
        [-0.2813,  0.2373, -0.0948, -0.3769,  0.3093,  0.1658]],
       grad_fn=<MmBackward0>)

In [147]:
d_k = keys.shape[-1] # embedding dimension of keys

In [148]:
queries @ keys.T / (d_k ** 0.5) # scaled dot product

tensor([[-0.2702,  1.1348, -0.2484, -0.7959,  0.8442,  0.5178],
        [-0.0905, -1.7994,  0.2873,  0.8071, -1.0502, -0.7015],
        [-0.3258,  0.5081, -0.2099, -0.3072,  0.3586,  0.1820],
        [-0.2819,  1.1671, -0.1519, -1.2444,  1.1164,  0.7126],
        [-0.1290, -0.4138,  0.0326,  0.1362, -0.2078, -0.1557],
        [-0.1624,  0.1370, -0.0548, -0.2176,  0.1786,  0.0957]],
       grad_fn=<DivBackward0>)

In [149]:
attention_weights = torch.softmax(queries @ keys.T / (d_k ** 0.5), dim=-1) # attention weights
attention_weights

tensor([[0.0838, 0.3415, 0.0856, 0.0495, 0.2554, 0.1842],
        [0.1661, 0.0301, 0.2424, 0.4076, 0.0636, 0.0902],
        [0.1100, 0.2533, 0.1236, 0.1121, 0.2182, 0.1828],
        [0.0739, 0.3147, 0.0842, 0.0282, 0.2992, 0.1998],
        [0.1632, 0.1227, 0.1918, 0.2127, 0.1508, 0.1589],
        [0.1406, 0.1897, 0.1566, 0.1331, 0.1978, 0.1821]],
       grad_fn=<SoftmaxBackward0>)

In [150]:
attention_weights @ values # context vectors

tensor([[ 0.3954, -0.3759,  0.1896],
        [-0.3271, -0.1062,  0.4188],
        [ 0.1751, -0.2676,  0.1961],
        [ 0.3553, -0.3115,  0.1510],
        [-0.1715, -0.1228,  0.1987],
        [-0.0089, -0.1786,  0.1498]], grad_fn=<MmBackward0>)

In [151]:
class AttentionWithTrainableWeights(nn.Module):
    def __init__(self):
        super().__init__()
        self.context_length = config["context_length"]
        self.dim_in = config["dim_in"]
        self.dim_out = config["dim_out"]
        self.W_key = nn.Linear(self.dim_in, self.dim_out)
        self.W_query = nn.Linear(self.dim_in, self.dim_out)
        self.W_value = nn.Linear(self.dim_in, self.dim_out)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.T
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim=-1)
        context_vectors = attention_weights @ values
        return context_vectors

In [152]:
attention = AttentionWithTrainableWeights()

In [153]:
attention(input_embeddings)

tensor([[ 0.4977, -0.6632, -0.4832],
        [ 0.4330, -0.6002, -0.3458],
        [ 0.3831, -0.6438, -0.3315],
        [ 0.9085, -0.1363,  0.5123],
        [ 0.2615, -0.7066, -0.4836],
        [ 0.3544, -0.6556, -0.3950]], grad_fn=<MmBackward0>)